# Smell Identification
## https://www.leibniz-lsb.de/en/datenbanken/leibniz-lsbtum-odorant-database/odorantdb

In [1]:
from __future__ import annotations
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import re
import random
import pandas as pd
import numpy as np

INPUT_CSV  = Path("Data/Occurrence_to_Name_IsomericSMILES.csv")
OUTPUT_CSV = Path("smell_identification_30.csv")
RANDOM_SEED = 42

TARGET_ITEMS = [
    "mango","peanut","hazelnut","tomato","apple","walnut","raspberry","peach","honey",
    "parsley","grapefruit","pineapple","strawberry","apricot","rice","grape","popcorn",
    "orange","cheese","melon","leather","chocolate","coffee","onion","fish","beer",
    "whisky","red wine","prawn","bread"
]

ALIASES = {
    "whisky": {"whisky", "whiskey"},
    "prawn": {"prawn", "prawns", "shrimp"},
    "peanut": {"peanut", "peanuts"},
    "hazelnut": {"hazelnut", "hazelnuts"},
    "raspberry": {"raspberry", "raspberries"},
    "grape": {"grape", "grapes"},
    "tomato": {"tomato", "tomatoes"},
    "cheese": {"cheese", "cheeses"},
    "fish": {"fish", "seafood"},
    "red wine": {"red wine", "redwine"},
}

NON_ALNUM = re.compile(r"[^a-z0-9]+")

def canon_item(s: str) -> str:
    """Lowercase, strip, collapse non-alnum to single space."""
    s = (s or "").strip().casefold()
    s = NON_ALNUM.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def build_target_synonyms(targets: List[str]) -> Dict[str, set]:
    syns: Dict[str, set] = {}
    for t in targets:
        base = canon_item(t)
        syn = {base}
        # simple plural/singular toggles
        if base.endswith("y"): syn.add(base[:-1] + "ies")
        if base.endswith("ies"): syn.add(base[:-3] + "y")
        if not base.endswith("s"): syn.add(base + "s")
        if base.endswith("s"): syn.add(base[:-1])
        # add manual aliases
        for a in ALIASES.get(t, set()):
            syn.add(canon_item(a))
        syns[t] = syn
    return syns

def parse_name_smiles(cell: str) -> Optional[Tuple[str, str]]:

    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return None
    s = str(cell).strip().strip('"').strip()
    if not s:
        return None
    if "," not in s:
        return None  # malformed
    left, right = s.rsplit(",", 1)
    name = left.strip().strip('"')
    smi  = right.strip().strip('"')
    if not smi or smi.upper() == "N/A":
        return None
    return (name, smi)

def load_items_table(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"Input not found: {path}")
    df = pd.read_csv(path, dtype=str)
    cas_cols = [c for c in df.columns if str(c).startswith("CAS_")]
    if "Item" not in df.columns or not cas_cols:
        raise SystemExit("Expected columns: 'Item' and at least one 'CAS_*' column.")
    return df[["Item"] + cas_cols].copy()

def collect_molecules_for_targets(df: pd.DataFrame, targets: List[str]) -> Dict[str, List[Tuple[str, str]]]:
    target_syns = build_target_synonyms(targets)
    item_rows = {}
    for idx, val in df["Item"].items():
        item_rows.setdefault(canon_item(val), []).append(idx)

    result: Dict[str, List[Tuple[str, str]]] = {}

    for t in targets:
        want = target_syns[t]
        hit_idxs = []
        for key, rows in item_rows.items():
            if key in want:
                hit_idxs.extend(rows)

        mols: List[Tuple[str, str]] = []
        seen_smiles = set()
        if hit_idxs:
            cas_cols = [c for c in df.columns if str(c).startswith("CAS_")]
            for ridx in hit_idxs:
                for c in cas_cols:
                    val = df.at[ridx, c]
                    parsed = parse_name_smiles(val)
                    if not parsed:
                        continue
                    name, smi = parsed
                    if smi not in seen_smiles:
                        seen_smiles.add(smi)
                        mols.append((name, smi))
        result[t] = mols
    return result

def build_questions(maps: Dict[str, List[Tuple[str, str]]], targets: List[str]) -> pd.DataFrame:
    rng = random.Random(RANDOM_SEED)
    rows = []
    qid = 1
    for item in targets:
        mols = maps.get(item, [])
        if not mols:
            print(f"[WARN] No molecules found for item: {item}")
            continue

        names  = ", ".join(n for n, _ in mols)
        smiles = ", ".join(s for _, s in mols)

        distract_pool = [x for x in targets if x != item and maps.get(x)]
        if len(distract_pool) < 3:
            distract_pool = [x for x in targets if x != item]
        distractors = rng.sample(distract_pool, k=min(3, len(distract_pool)))
        opts = distractors + [item]
        rng.shuffle(opts)

        opts_str = ";".join(opts)  

        prompt1 = (
            f"Given the molecules {smiles}, select exactly one item from {opts_str} "
            f"whose odor profile most likely includes these molecules. Reply with the item name only. "
            f"Do not write any comments. DO NOT SEARCH ONLINE"
        )
        prompt2 = (
            f"Given the molecules {names}, select exactly one item from {opts_str} "
            f"whose odor profile most likely includes these molecules. Reply with the item name only. "
            f"Do not write any comments. DO NOT SEARCH ONLINE."
        )

        rows.append({
            "question_ID": qid,
            "compound.name_1": names,
            "SMILES_1": smiles,
            "compound.name_2": "",
            "SMILES_2": "",
            "OPTIONS": opts_str,
            "question_category": "smell_identification",
            "prompt.1": prompt1,
            "prompt.2": prompt2,
           "answer": item,
            "other_info": "",
        })
        qid += 1

    return pd.DataFrame(rows)

def main():
    df = load_items_table(INPUT_CSV)
    maps = collect_molecules_for_targets(df, TARGET_ITEMS)
    out = build_questions(maps, TARGET_ITEMS)

    if out.empty:
        raise SystemExit("No questions produced (no target items found).")

    out.to_csv(OUTPUT_CSV, index=False)
    print(f"[OK] Wrote {OUTPUT_CSV.resolve()} with {len(out)} rows.")

if __name__ == "__main__":
    main()


[OK] Wrote /Users/vs479/Desktop/Desktop/HNL/OlfactoryInteligence_Benchmark/7-Hard_2/smell_identification_30.csv with 30 rows.
